## Stream Customers Data From Cloud Files to Delta Lake using Autoloader

1. Read files from cloud storage using Autoloader
2. Transform the dataframe to add the following columns
    - file path: Cloud file path
    - ingestion date: Current timestamp
3. Write the transformed data stream to Delta Lake Table

## 1. Read files using Auto Loader

In [0]:
customers_df = (spark.readStream 
     .format('cloudFiles')
     .option("cloudFiles.format","json")
     .option("cloudFiles.schemaLocation","abfss://gizmobox@deacourseexternaldl.dfs.core.windows.net/landing/operational_data/customers_autoloader/_schema")
     .option("cloudFiles.inferColumnTypes","true")
     .option("cloudFiles.schemaHints","date_of_birth DATE, member_since DATE,created_timestamp TIMESTAMP")
     .option("cloudFiles.schemaEvolutionMode","rescue") ## other options "addNEwColumns","rescue","failOnNewColumns","none","addNewColumnsWithTypeWidening"
     .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/") 
     )


## 2. Transform the dataframe to add the following columns
        - file path: Cloud file path
        - ingestion date: Current timestamp

In [0]:
from pyspark.sql import functions as F

cunstomers_transformed_df = (customers_df.withColumn("file_path",F.col("_metadata.file_path"))
            .withColumn("ingestion_date", F.current_timestamp())
)

## 3. Write the transformed data stream to Delta Lake Table

In [0]:
streaming_query = (
    cunstomers_transformed_df.writeStream 
        .format("delta") 
        .option("checkpointLocation", "abfss://gizmobox@deacourseexternaldl.dfs.core.windows.net/landing/operational_data/customers_autoloader/_checkpoint_stream")
        .option("mergeSchema","true")
        .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
%fs ls 'abfss://gizmobox@deacourseexternaldl.dfs.core.windows.net/landing/operational_data/'

In [0]:
%sql
select * from gizmobox.bronze.customers_autoloader

In [0]:
streaming_query.stop()

In [0]:
%sql
drop table gizmobox.bronze.customers_autoloader